In [ ]:
import io
import copy
import os
import pandas as pd
import sys
import string
import re

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

###  import archaic dervied MPRA results

In [ ]:
MPRA_results = pd.read_csv(f'humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                         #usecols=range(0, 34), 
                         header=0)

### Important

#### WTF always calculated REF - ALT. Therefor I want to get 

#### REF - ALT = Derived - Ancestral = Modern - Ancestral

#### So the varaint must be Modern -> Ancestral

FIMO calculates the score as REF-ALT. Since I want to get positive values as one where the modern allele is binding more than archaic allele, I set the variant as alt (=modern) -> ref (=ancestral). In the FIMO config file I set assembly_type=ref

In [ ]:
# Split entries on semicolon and explode
split_MPRA_results = MPRA_results["variants_genomic"].fillna("").astype(str).str.split(";").explode().str.strip()

# Regex to capture chr, pos, ref, alt
pat = re.compile(r"^(chr[\w]+):(\d+)\(([A-Za-z]+)->([A-Za-z]+)\)$")
parsed = split_MPRA_results.str.extract(pat)
parsed.columns = ["chr", "pos", "ape", "human"]

# Drop rows where parsing failed. Then drop duplicates
parsed = (
    parsed
    .dropna(subset=["chr", "pos", "ape", "human"])
    .drop_duplicates()
)
# Cast pos to int
parsed["pos"] = parsed["pos"].astype(int)

# Build output DataFrame
FIMO_input = pd.DataFrame({
    "chr": parsed["chr"],
    "start": parsed["pos"]-1,
    "end": parsed["pos"],  # one base ahead for SNVs
    "variant": parsed["human"] + "->" + parsed["ape"]
}).reset_index(drop=True)

print(FIMO_input)


In [ ]:
FIMO_input.to_csv("humanMPRA/TF_analysis/final/variant_tool_input/hMPRA_variant_tool_input_chonds.bed", sep='\t', header=False, index=False)